In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def promedio_12_meses_780p():

    df = pd.read_csv("../../data/preprocessed/periodo_x_producto_con_target.csv", sep=',', encoding='utf-8')
    df = df[df['periodo'] >= 201901]  # Filtrar desde 201901

    productos_ok = pd.read_csv("../../data/raw/product_id_apredecir201912.csv", sep="\t")

    df = df.merge(productos_ok, on='product_id', how='inner')

    df = df.groupby('product_id').agg({'tn': 'mean'}).reset_index()

    return df

promedio = promedio_12_meses_780p()
promedio

,product_id,tn
0,20001,1454.732720
1,20002,1175.437142
2,20003,784.976407
3,20004,627.215328
4,20005,668.270104
...,...,...
775,21263,0.029993
776,21265,0.089541
777,21266,0.094659
778,21267,0.092835


In [ ]:
autoarima = pd.read_csv("./outputs/autoarima.csv", sep=',')
autoarima.rename(columns={'prediccion_mes+2':'tn_arima'}, inplace=True)

In [ ]:
prophet = pd.read_csv("./outputs/prophet.csv", sep=',')
prophet.rename(columns={'yhat':'tn_prophet'}, inplace=True)

# 1. Hacemos un merge entre los dos dataframes para tener tn y tn_prophet juntos
df = prophet.merge(promedio, on='product_id', how='left')  # crea columnas tn_prophet y tn

mask = df['tn_prophet'] < 0
df.loc[mask, 'tn_prophet'] = df.loc[mask, 'tn']

# 3. (opcional) eliminamos la columna 'tn' si ya no la necesitás
df.drop(columns=['tn'], inplace=True)

prophet = df[['product_id','tn_prophet']]

In [ ]:
neuralprophet = pd.read_csv("./outputs/neuralprophet.csv", sep=',')
neuralprophet.rename(columns={'yhat1':'tn_neuralprophet'}, inplace=True)

# 1. Hacemos un merge entre los dos dataframes para tener tn y tn_prophet juntos
df = neuralprophet.merge(promedio, on='product_id', how='left')  # crea columnas tn_prophet y tn

mask = df['tn_neuralprophet'] < 0
df.loc[mask, 'tn_neuralprophet'] = df.loc[mask, 'tn']

# 3. (opcional) eliminamos la columna 'tn' si ya no la necesitás
df.drop(columns=['tn'], inplace=True)

neuralprophet = df[['product_id', 'tn_neuralprophet']].copy()


In [ ]:
autoarima = autoarima.merge(prophet, on='product_id', how='left')
autoarima = autoarima.merge(neuralprophet, on='product_id', how='left')
autoarima

,product_id,tn_arima,tn_prophet,tn_neuralprophet
0,20001,1516.433668,1362.070852,978.498200
1,20002,1389.763832,1117.738591,1085.884400
2,20003,726.958620,624.334799,552.694200
3,20004,572.913140,230.298529,406.699340
4,20005,568.827329,279.618412,541.698850
...,...,...,...,...
775,21263,0.014009,0.063489,0.029993
776,21265,0.065632,0.089541,0.067555
777,21266,0.070981,0.094659,0.074221
778,21267,0.032159,0.092835,0.019894


In [ ]:
autoarima['tn'] = (autoarima['tn_arima'] + autoarima['tn_prophet'] + autoarima['tn_neuralprophet'])/3
autoarima[['product_id', 'tn']].to_csv("./outputs/ensemble_arima_prophet_nprophet.csv", sep=',', index=False)

In [ ]:
autogluon = pd.read_csv("./outputs/prediccion_autogluon_2ventanas.csv", sep=',')
autogluon.rename(columns={'tn':'tn_ag'}, inplace=True)

In [ ]:
reg_lineal = pd.read_csv("./outputs/predicciones_regresion_lineal_v1.csv", sep=',')
reg_lineal.rename(columns={'tn':'tn_reg_lineal'}, inplace=True)

In [ ]:
autoarima = autoarima.merge(autogluon, on='product_id', how='left')
autoarima = autoarima.merge(reg_lineal, on='product_id', how='left')

In [ ]:
autoarima['tn'] = (autoarima['tn_arima'] + autoarima['tn_prophet'] + autoarima['tn_neuralprophet'] + autoarima['tn_ag'] + autoarima['tn_reg_lineal'])/5
autoarima[['product_id', 'tn']].to_csv("./outputs/ensemble_arima_prophet_nprophet_ag_reglineal.csv", sep=',', index=False)

In [ ]:
autoarima[['product_id', 'tn']]

,product_id,tn
0,20001,1264.506278
1,20002,1172.876970
2,20003,654.345423
3,20004,462.561618
4,20005,496.567492
...,...,...
775,21263,0.121116
776,21265,0.073439
777,21266,0.078433
778,21267,0.059325


| PRODUCT\_ID | TN        |
| ----------- | --------- |
| 20001       | 1122.1555 |
| 20006       | 749.1662  |
| 20007       | 401.1913  |
| 20012       | 224.8046  |
| 20017       | 242.0700  |
| 20018       | 386.8463  |
| 20032       | 352.3456  |
| 20033       | 121.0774  |
| 20043       | 189.5840  |

In [ ]:
autoarima.loc[autoarima['product_id'] == 20001, 'tn'] = 1122.1555
autoarima.loc[autoarima['product_id'] == 20006, 'tn'] = 749.1662
autoarima.loc[autoarima['product_id'] == 20007, 'tn'] = 401.1913
autoarima.loc[autoarima['product_id'] == 20012, 'tn'] = 224.8046
autoarima.loc[autoarima['product_id'] == 20017, 'tn'] = 242.0700
autoarima.loc[autoarima['product_id'] == 20018, 'tn'] = 386.8463
autoarima.loc[autoarima['product_id'] == 20032, 'tn'] = 352.3456
autoarima.loc[autoarima['product_id'] == 20033, 'tn'] = 121.0774
autoarima.loc[autoarima['product_id'] == 20043, 'tn'] = 189.5840


In [ ]:
autoarima[['product_id', 'tn']].to_csv("./outputs/ensemble_arima_prophet_nprophet_ag_reglineal.csv", sep=',', index=False)

In [ ]:
promedio[promedio['product_id']==20017]

,product_id,tn
16,20017,287.385752
